In [ ]:
import os
import time
import base64
import ctypes
import requests
import schedule
import pyautogui
import pygetwindow as gw
from PIL import Image
from dotenv import load_dotenv

load_dotenv()

# Força captura em pixels físicos reais (ignora DPI scaling do Windows)
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(2)  # PROCESS_PER_MONITOR_DPI_AWARE
except Exception:
    try:
        ctypes.windll.user32.SetProcessDPIAware()
    except Exception:
        pass

# ============================================================
# CONFIGURAÇÕES - PREENCHA AQUI
# ============================================================
PASTA_DESTINO      = r"C:\Users\LucasCavalcante\Documents\Prints_GARE11"
NOME_JANELA        = 'Broadcast'
NOME_ARQUIVO_PRINT = 'print_PARCIAL_GARE11.png'
ESCALA_UPSCALE     = 2   # 2 = dobra a resolução; use 1 para desativar

# Z-API
ZAPI_INSTANCE_ID   = os.getenv("INSTANCE_ID")
ZAPI_TOKEN         = os.getenv("TOKEN_Z")
ZAPI_CLIENT_TOKEN  = os.getenv("CLIENT_TOKEN")
GRUPO_PHONE        = os.getenv("GRUPO_PHONE")   # adicione esta variável no .env após obter o phone do grupo

# CLAUDE (Anthropic)
ANTHROPIC_API_KEY  = os.getenv("CLAUDE_API")

PROMPT_ANALISE = (
"Você é um analista financeiro especializado em mercado de capitais brasileiro, "
"com foco em Fundos Imobiliários (FIIs).\n\n"
"Sua tarefa é analisar um print de tela do sistema de 'Players' de uma corretora "
"(estilo Profit/Nelogica), que mostra as operações de compra e venda de corretoras "
"em um determinado ativo durante o pregão.\n\n"
"A imagem contém:\n"
"- Painel superior: book de ofertas com cotação atual do ativo e variação percentual.\n"
"- Painel inferior ESQUERDO: quadro de COMPRAS — tabela 'Mercado Regular' com colunas "
"Corretora e Vtt (volume financeiro de compras). Na linha 'Totais' (última linha), "
"o campo Vtt representa o volume financeiro total negociado no pregão até o momento.\n"
"- Painel inferior DIREITO: quadro de VENDAS — tabela 'Mercado Regular' com colunas "
"Corretora, Vtt (volume financeiro de vendas) e Saldo Vtt. São quadros separados e "
"independentes — cada um lista suas próprias corretoras e volumes.\n\n"
"CONVENÇÃO DE ESCALA OBRIGATÓRIA:\n"
"- K = milhares (ex.: 524K = R$ 524.000)\n"
"- M = milhões (ex.: 1,3M = R$ 1.300.000)\n"
"- Portanto, qualquer valor em M é sempre maior que qualquer valor em K.\n"
"- Preserve sempre a escala original da imagem. Nunca converta K para M nem o contrário.\n\n"
"ETAPA OBRIGATÓRIA ANTES DE GERAR O RESUMO — VARREDURA INTERNA DA TABELA:\n"
"Esta etapa é um processo de raciocínio interno. NÃO escreva a varredura na resposta. "
"NÃO exiba tabelas, listas ou qualquer output desta etapa. "
"Use os resultados apenas para construir o resumo final.\n\n"
"Internamente, execute os seguintes passos:\n\n"
"PASSO 1 — Leia linha por linha a coluna Vtt do quadro de COMPRAS (painel inferior "
"esquerdo), do topo até a linha Totais. Para cada linha, registre: nome da corretora "
"e valor Vtt de Compras.\n\n"
"PASSO 2 — Leia linha por linha a coluna Vtt do quadro de VENDAS (painel inferior "
"direito), do topo até a linha Totais. Para cada linha, registre: nome da corretora "
"e valor Vtt de Vendas.\n\n"
"PASSO 3 — Ordene mentalmente os dois conjuntos de valores, convertendo tudo para "
"a mesma unidade antes de comparar (ex.: 1,3M = 1300K). Somente após essa ordenação "
"completa, identifique os TOP 3 de cada lado.\n\n"
"PASSO 4 — Verificação cruzada: os dois quadros são independentes. É esperado e normal "
"que as corretoras do TOP 3 Compradores sejam DIFERENTES das do TOP 3 Vendedores. "
"Se os dois rankings contiverem exatamente as mesmas corretoras na mesma ordem, "
"você cometeu um erro de leitura — releia cada quadro separadamente antes de prosseguir.\n\n"
"ATENÇÃO: a tabela NÃO está necessariamente ordenada por volume. Os maiores "
"compradores e vendedores podem aparecer em qualquer linha. Nunca assuma que as "
"primeiras linhas contêm os maiores valores — percorra TODAS as linhas de cada quadro.\n\n"
"O primeiro e único output visível deve ser o resumo formatado para WhatsApp, "
"começando diretamente pelo emoji 📊 do cabeçalho.\n\n"
"Gere um resumo objetivo, claro e formatado para WhatsApp, contendo OBRIGATORIAMENTE "
"as seguintes seções nesta ordem:\n\n"
"1. Cabeçalho: Nome do ativo, data e horário da atualização.\n"
"2. Cotação: Preço atual da cota, variação percentual do dia e volume financeiro total "
"negociado (Vtt da linha Totais do quadro de Compras).\n"
"3. TOP 3 VENDEDORES: as 3 corretoras com maior Vtt no quadro de Vendas, ordenadas do "
"maior para o menor.\n"
"4. TOP 3 COMPRADORES: as 3 corretoras com maior Vtt no quadro de Compras, ordenadas do "
"maior para o menor.\n"
"REGRAS DE FORMATAÇÃO (WhatsApp):\n"
"- Use emojis para identificação visual: 📊 (cabeçalho), 💰 (cotação), 🔴 (vendedores), "
"🟢 (compradores).\n"
"- Use *asteriscos simples* para negrito (padrão WhatsApp).\n"
"- Liste os rankings em formato vertical, um item abaixo do outro, usando os emojis "
"numéricos: 1️⃣ 2️⃣ 3️⃣.\n"
"- Separe as seções com linhas horizontais usando '---'.\n"
"- Mantenha os valores no formato brasileiro (vírgula decimal, K para milhares, M para "
"milhões), exatamente como aparecem na imagem.\n"
"- O resumo final deve conter APENAS os valores corretos após a análise completa. "
"Nunca mencione revisões, correções, versões anteriores ou mudanças de valores. "
"Não use expressões como 'corrigindo', 'revisando', 'anteriormente', 'na verdade', "
"'reanalise' ou qualquer linguagem que indique que houve uma correção. "
"Entregue diretamente o resultado final como se fosse a primeira e única análise.\n\n"
"REGRAS DE PRECISÃO:\n"
"- Leia os valores da imagem com atenção. Não invente, não arredonde além do que está "
"exibido.\n"
"- O ranking de Vendedores deve usar APENAS a coluna Vtt do quadro de Vendas.\n"
"- O ranking de Compradores deve usar APENAS a coluna Vtt do quadro de Compras.\n"
"- Os dois quadros são fisicamente separados na imagem — nunca misture os valores de um "
"com os do outro.\n"
"- O volume total (seção 2) deve ser lido da linha Totais do quadro de Compras, "
"não somado manualmente.\n"
"- Se algum dado estiver ilegível na imagem, indique '[ilegível]' em vez de adivinhar.\n"
"- Se mais de 30%% da tabela estiver ilegível, sinalize no cabeçalho antes de prosseguir.\n"
"- Responda sempre em português brasileiro.\n"
)
# ============================================================


def garantir_pasta_existente(pasta):
    os.makedirs(pasta, exist_ok=True)


def encontrar_janela(nome_janela):
    for w in gw.getWindowsWithTitle(nome_janela):
        if nome_janela.upper() in w.title.upper():
            return w
    return None


def preparar_janela_para_print(janela):
    try:
        janela.minimize()
        time.sleep(1)
        janela.restore()
        time.sleep(2)
    except Exception as e:
        print(f"⚠️ Erro ao restaurar a janela: {e}")


def tirar_screenshot_da_janela(janela, pasta_destino, nome_arquivo):
    try:
        if janela.left == 0 and janela.top == 0 and janela.width == 0 and janela.height == 0:
            raise Exception("Coordenadas inválidas da janela")

        x, y = janela.left, janela.top
        w_, h_ = janela.width, janela.height
        caminho_arquivo = os.path.join(pasta_destino, nome_arquivo)

        screenshot = pyautogui.screenshot(region=(x, y, w_, h_))

        if ESCALA_UPSCALE > 1:
            novo_w = screenshot.width * ESCALA_UPSCALE
            novo_h = screenshot.height * ESCALA_UPSCALE
            screenshot = screenshot.resize((novo_w, novo_h), Image.LANCZOS)
            print(f"🔍 Resolução após upscale: {novo_w}x{novo_h}px")

        screenshot.save(caminho_arquivo, format="PNG", optimize=False, compress_level=1)

        print(f"✅ Screenshot salvo em: {caminho_arquivo}")
        return caminho_arquivo

    except Exception as e:
        print(f"❌ Erro ao tirar o print: {e}")
        return None


def analisar_imagem_com_claude(caminho_imagem):
    print("🤖 Analisando imagem com Claude...")

    with open(caminho_imagem, "rb") as f:
        imagem_base64 = base64.standard_b64encode(f.read()).decode("utf-8")

    headers = {
        "x-api-key": ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    }

    payload = {
        "model": "claude-opus-4-7",
        "max_tokens": 1024,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": imagem_base64,
                        },
                    },
                    {
                        "type": "text",
                        "text": PROMPT_ANALISE
                    }
                ],
            }
        ],
    }

    response = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers=headers,
        json=payload
    )

    if response.status_code == 200:
        resumo = response.json()["content"][0]["text"]
        print(f"✅ Resumo gerado pelo Claude:\n{resumo}")
        return resumo
    else:
        print(f"❌ Erro na API do Claude: {response.status_code} - {response.text}")
        return None


def enviar_imagem_whatsapp(caminho_imagem, caption):
    print("📲 Enviando imagem para o WhatsApp...")

    with open(caminho_imagem, "rb") as f:
        imagem_base64 = base64.standard_b64encode(f.read()).decode("utf-8")

    imagem_base64_formatada = f"data:image/png;base64,{imagem_base64}"

    url = f"https://api.z-api.io/instances/{ZAPI_INSTANCE_ID}/token/{ZAPI_TOKEN}/send-image"

    headers = {
        "Client-Token": ZAPI_CLIENT_TOKEN,
        "Content-Type": "application/json",
    }

    payload = {
        "phone": GRUPO_PHONE,
        "image": imagem_base64_formatada,
        "caption": caption,
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        print(f"✅ Imagem enviada com sucesso para o grupo!")
        print(f"   messageId: {response.json().get('messageId')}")
    else:
        print(f"❌ Erro ao enviar via Z-API: {response.status_code} - {response.text}")


def tarefa_automatica():
    print("\n" + "="*50)
    print("🕐 Iniciando rotina de captura + análise + envio...")
    print("="*50)

    garantir_pasta_existente(PASTA_DESTINO)

    janela = encontrar_janela(NOME_JANELA)
    if not janela:
        print(f"❌ Janela '{NOME_JANELA}' não encontrada.")
        return

    preparar_janela_para_print(janela)
    caminho_print = tirar_screenshot_da_janela(janela, PASTA_DESTINO, NOME_ARQUIVO_PRINT)
    if not caminho_print:
        return

    resumo = analisar_imagem_com_claude(caminho_print)
    if not resumo:
        return

    enviar_imagem_whatsapp(caminho_print, caption=resumo)


schedule.every(60).minutes.do(tarefa_automatica)

tarefa_automatica()

while True:
    schedule.run_pending()
    time.sleep(5)